# ASVspoof 2019 PA — Real-code pipeline smoke test

This notebook exercises the **existing project implementation** on a small, balanced subset of the official train/dev/eval manifests. It is intended to catch integration errors before the full reproducibility run.

It runs the actual feature extractors, frame-cache aggregation, fusion, E1–E7 XGBoost training, PA min-tDCF scripts, one-trial E7_FINAL tuning, final evaluation, global/class-conditional/individual SHAP, and condition-analysis scripts. It does **not** run the full-corpus workload or the production 80-trial/10,000-estimator search.

All generated files go to a unique directory under `results/`; the existing `data/`, `results/`, model outputs and feature caches are not overwritten. The test configuration is temporary and does not modify `configs/data.yaml`.

**Important:** metrics from this subset are engineering smoke-test outputs only. They are not ASVspoof benchmark results and must not be used in a paper. A successful smoke test proves that the code paths execute on the sampled data; it does not guarantee that the full job will fit available disk/RAM or finish successfully.

## 1. Configure paths and test size

Run this notebook from the project root or its `notebooks/` directory. `DATASET_ROOT` must be the directory containing `PA/`. The ASV score directory can be overridden separately. Use the same XGBoost device intended for the full run.

In [ ]:
import os
import sys
import json
import time
import shutil
import tempfile
import subprocess
import platform
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

DATASET_ROOT = Path(os.environ.get("ASVSPOOF_DATASET_ROOT", "/path/to/ASVspoof2019_archive")).expanduser()
ASV_SCORE_DIR = Path(
    os.environ.get("ASVSPOOF_ASV_SCORE_DIR", str(DATASET_ROOT / "PA" / "ASVspoof2019_PA_asv_scores"))
).expanduser()
PROJECT_ROOT_OVERRIDE = os.environ.get("ASVSPOOF_PROJECT_ROOT") or None

# This smoke test intentionally reduces workload; the full notebook uses the production settings.
SAMPLES_PER_CLASS_PER_SPLIT = 500
EXTRACTION_WORKERS = 8
XGB_DEVICE = "cuda"  # Set to "cpu" if this environment does not have CUDA-enabled XGBoost.
SMOKE_N_ESTIMATORS = 50*3
SMOKE_EARLY_STOPPING = 8*3
SMOKE_TUNING_TRIALS = 1*3
SMOKE_TUNING_N_ESTIMATORS = 80*3
SMOKE_TUNING_EARLY_STOPPING = 8
RANDOM_SEED = 42
RUN_PAIRED_CONDITION_STATS = True
RUN_INDIVIDUAL_SHAP = True

print("Python:", sys.version.split()[0])
print("Kernel:", sys.executable)
print("Platform:", platform.platform())
print("Dataset root:", DATASET_ROOT)
print("ASV score directory:", ASV_SCORE_DIR)
print("Requested XGBoost device:", XGB_DEVICE)
print("Samples per class per split:", SAMPLES_PER_CLASS_PER_SPLIT)

## 2. Locate the complete project and check required source files

This check intentionally requires `src/data/` and `src/features/`. They are implementation code, not optional dataset folders.

In [ ]:
def find_project_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "src" / "cli.py").is_file() and (candidate / "tools" / "tune_e7.py").is_file():
            return candidate
    raise FileNotFoundError(
        "Could not locate project root. Set ASVSPOOF_PROJECT_ROOT to the folder containing src/cli.py and tools/tune_e7.py."
    )

PROJECT_ROOT = Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve() if PROJECT_ROOT_OVERRIDE else find_project_root(Path.cwd())
if not PROJECT_ROOT.is_dir():
    raise NotADirectoryError(PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

REQUIRED_FILES = [
    "requirements.txt", "pyproject.toml", "configs/data.yaml", "src/cli.py", "src/config.py",
    "src/data/audio.py", "src/data/manifest.py", "src/data/protocol.py", "src/data/validation.py",
    "src/features/cqt.py", "src/features/tkeo.py", "src/features/acoustic.py",
    "src/pipeline/cqcc.py", "src/pipeline/tkeo.py", "src/pipeline/acoustic.py",
    "src/pipeline/fusion.py", "src/pipeline/cache.py", "src/models/xgb_fusion.py",
    "tools/tune_e7.py", "tools/evaluate_tuned_e7.py", "tools/compute_pa_tdcf.py",
    "tools/integrate_tdcf_results.py", "tools/shap_global_e7.py",
    "tools/shap_class_conditional_e7.py", "tools/explain_one_trial.py",
    "tools/ASVspoof_PA_condition_analysis.py", "tools/paired_pa_condition_stats.py",
    "tools/tdcf/eval_metrics.py",
]
missing = [name for name in REQUIRED_FILES if not (PROJECT_ROOT / name).is_file()]
if missing:
    raise FileNotFoundError("Incomplete source tree; missing:\n - " + "\n - ".join(missing))
print("Project root:", PROJECT_ROOT)
print(f"PASS: all {len(REQUIRED_FILES)} required source files are present.")

## 3. Check packages and XGBoost runtime

The test uses the selected `XGB_DEVICE` for the seven ablation smoke models and the one-trial E7 tune. This is a real tiny training run, not only a build-information check.

In [ ]:
import xgboost as xgb
import sklearn
import scipy
import soundfile as sf
import shap
import matplotlib

print("XGBoost:", xgb.__version__)
print("XGBoost build:", xgb.build_info())
print("scikit-learn:", sklearn.__version__)
print("SciPy:", scipy.__version__)
print("SHAP:", shap.__version__)
print("Matplotlib:", matplotlib.__version__)

if XGB_DEVICE not in {"cpu", "cuda"}:
    raise ValueError("XGB_DEVICE must be 'cpu' or 'cuda'.")
if XGB_DEVICE == "cuda" and xgb.build_info().get("USE_CUDA") is False:
    raise RuntimeError("XGBoost reports USE_CUDA=False. Set XGB_DEVICE='cpu' or install CUDA-enabled XGBoost.")

rng = np.random.default_rng(RANDOM_SEED)
X_device = rng.normal(size=(64, 12)).astype(np.float32)
y_device = np.array([0, 1] * 32, dtype=np.int32)
probe = xgb.XGBClassifier(
    n_estimators=3, max_depth=2, tree_method="hist", device=XGB_DEVICE,
    n_jobs=1, eval_metric="logloss", verbosity=0, random_state=RANDOM_SEED,
)
probe.fit(X_device, y_device)
print(f"PASS: small XGBoost training completed on requested device={XGB_DEVICE!r}.")

## 4. Create an isolated smoke workspace and temporary data configuration

The smoke workspace is unique for each run. The project’s normal dataset config, manifests, feature cache, matrices and trained models are not touched.

In [ ]:
CONFIG_SOURCE = PROJECT_ROOT / "configs" / "data.yaml"
base_config = yaml.safe_load(CONFIG_SOURCE.read_text(encoding="utf-8"))
base_config["data"]["root"] = str(DATASET_ROOT.resolve())
base_config["data"]["access_dir"] = "PA"

PA_ROOT = DATASET_ROOT.resolve() / "PA"
ASV_SCORE_FILES = {
    "dev": ASV_SCORE_DIR.resolve() / "ASVspoof2019.PA.asv.dev.gi.trl.scores.txt",
    "eval": ASV_SCORE_DIR.resolve() / "ASVspoof2019.PA.asv.eval.gi.trl.scores.txt",
}

required_dataset = {}
for split in ("train", "dev", "eval"):
    required_dataset[f"{split} protocol"] = PA_ROOT / base_config["data"]["protocols"][split]
    required_dataset[f"{split} audio"] = PA_ROOT / base_config["data"]["audio_dirs"][split]
missing_dataset = [f"{label}: {path}" for label, path in required_dataset.items() if not path.exists()]
missing_scores = [str(p) for p in ASV_SCORE_FILES.values() if not p.is_file()]
if missing_dataset or missing_scores:
    raise FileNotFoundError(
        "Dataset/ASV files missing:\n - " + "\n - ".join(missing_dataset + missing_scores)
    )

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
SMOKE_ROOT = PROJECT_ROOT / "results" / f"_pipeline_smoke_{stamp}"
SMOKE_PROCESSED = SMOKE_ROOT / "data" / "processed"
SMOKE_MANIFESTS = SMOKE_PROCESSED / "manifests"
SMOKE_CACHE = SMOKE_PROCESSED / "cache"
SMOKE_GROUP_DIR = SMOKE_ROOT / "results" / "group_features"
SMOKE_FUSION_DIR = SMOKE_ROOT / "results" / "fusion_features"
SMOKE_ABLATION_DIR = SMOKE_ROOT / "results" / "xgb_ablation"
SMOKE_TDCF_DIR = SMOKE_ROOT / "results" / "tdcf"
SMOKE_TUNING_DIR = SMOKE_ROOT / "results" / "xgb_tuning" / "E7_FINAL"
SMOKE_MODEL_DIR = SMOKE_TUNING_DIR / "best_model"
SMOKE_GLOBAL_SHAP_DIR = SMOKE_ROOT / "results" / "explainability" / "E7_FINAL" / "global"
SMOKE_CLASS_SHAP_DIR = SMOKE_ROOT / "results" / "explainability" / "E7_FINAL" / "class_conditional"
SMOKE_INDIVIDUAL_SHAP_DIR = SMOKE_ROOT / "results" / "explainability" / "E7_FINAL" / "individual"

for path in (SMOKE_MANIFESTS, SMOKE_CACHE, SMOKE_GROUP_DIR, SMOKE_FUSION_DIR):
    path.mkdir(parents=True, exist_ok=True)

# load_config() derives the project root from configs/<file>.yaml, so keep this temporary config inside configs/.
fd, temp_cfg_name = tempfile.mkstemp(prefix=".data_smoke_", suffix=".yaml", dir=PROJECT_ROOT / "configs")
os.close(fd)
SMOKE_CONFIG_PATH = Path(temp_cfg_name)
base_config["output"]["processed_dir"] = str(SMOKE_PROCESSED)
base_config["output"]["manifest_dir"] = str(SMOKE_MANIFESTS)
base_config["output"]["validation_dir"] = str(SMOKE_PROCESSED / "validation")
base_config["output"]["cache_dir"] = str(SMOKE_CACHE)
SMOKE_CONFIG_PATH.write_text(yaml.safe_dump(base_config, sort_keys=False), encoding="utf-8")
CONFIG_ARG = str(SMOKE_CONFIG_PATH.relative_to(PROJECT_ROOT))

print("Smoke workspace:", SMOKE_ROOT)
print("Temporary config:", SMOKE_CONFIG_PATH)
print("ASV dev score:", ASV_SCORE_FILES["dev"])
print("ASV eval score:", ASV_SCORE_FILES["eval"])

## 5. Call the existing manifest command, then create balanced smoke manifests

This calls `python -m src.cli manifest`. The manifest files written for this test are isolated inside the smoke workspace. The subset is sampled separately for train, dev and eval and preserves the original manifest row order.

In [ ]:
def run_cli(*args):
    command = [sys.executable, "-m", "src.cli", *map(str, args)]
    print("\n$", " ".join(command), flush=True)
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)



def run_tool(script, *args):
    script_path = PROJECT_ROOT / script

    if not script_path.is_file():
        raise FileNotFoundError(script_path)

    command = [
        sys.executable,
        str(script_path),
        *map(str, args)
    ]

    print("\n$", " ".join(command), flush=True)

    result = subprocess.run(
        command,
        cwd=PROJECT_ROOT,
        text=True,
        capture_output=True
    )

    if result.stdout:
        print(result.stdout)

    if result.stderr:
        print("STDERR:\n", result.stderr)

    if result.returncode != 0:
        raise RuntimeError(
            f"{script} failed with exit code "
            f"{result.returncode}"
        )

    return result



run_cli("manifest", "--config", CONFIG_ARG, "--splits", "train", "dev", "eval")

smoke_manifest_summary = {}
for split in ("train", "dev", "eval"):
    manifest_path = SMOKE_MANIFESTS / f"PA_{split}.csv"
    full_df = pd.read_csv(manifest_path, dtype={"utterance_id": str})
    if not {"bonafide", "spoof"}.issubset(set(full_df["key"].astype(str))):
        raise ValueError(f"{split} manifest does not contain both classes.")
    selected_indices = []
    for key in ("bonafide", "spoof"):
        candidates = full_df.index[full_df["key"].astype(str) == key].to_numpy()
        if len(candidates) < SAMPLES_PER_CLASS_PER_SPLIT:
            raise ValueError(
                f"{split} has only {len(candidates)} {key} trials; need {SAMPLES_PER_CLASS_PER_SPLIT}."
            )
        selected_indices.extend(
            np.random.default_rng(RANDOM_SEED + len(split) + (0 if key == "bonafide" else 100)).choice(
                candidates, size=SAMPLES_PER_CLASS_PER_SPLIT, replace=False
            ).tolist()
        )
    subset = full_df.loc[sorted(selected_indices)].reset_index(drop=True)
    if subset["audio_path"].map(lambda p: Path(str(p)).is_file()).sum() != len(subset):
        raise FileNotFoundError(f"At least one selected {split} audio path is missing.")
    subset.to_csv(manifest_path, index=False)
    smoke_manifest_summary[split] = {
        "n": int(len(subset)),
        "bonafide": int((subset["key"] == "bonafide").sum()),
        "spoof": int((subset["key"] == "spoof").sum()),
    }
    print(split.upper(), smoke_manifest_summary[split])

print("PASS: balanced train/dev/eval smoke manifests created.")

## 6. Run all three real feature extractors and the real frame-to-utterance aggregation

The commands and implementation are the same `src.cli` subcommands used by the full workflow. The only differences are the small manifest, one extraction worker, and an isolated cache/output directory.

In [ ]:
FEATURE_GROUPS = ("cqcc", "tkeo", "acoustic")
EXTRACT_COMMANDS = {"cqcc": "extract-cqcc", "tkeo": "extract-tkeo", "acoustic": "extract-acoustic"}
AGGREGATION_STATS = ("mean", "std", "p10", "p50", "p90")
EXPECTED_GROUP_WIDTHS = {"cqcc": 450, "tkeo": 600, "acoustic": 300}

extraction_reports = {}
for split in ("train", "dev", "eval"):
    expected_rows = smoke_manifest_summary[split]["n"]
    for group in FEATURE_GROUPS:
        start = time.time()
        run_cli(
            EXTRACT_COMMANDS[group], "--config", CONFIG_ARG, "--splits", split,
            "--workers", EXTRACTION_WORKERS, "--cache-root", SMOKE_CACHE, "--overwrite",
        )
        report_path = SMOKE_PROCESSED / "feature_reports" / f"{group}_{split}_summary.json"
        report = json.loads(report_path.read_text(encoding="utf-8"))
        if report.get("failed_count", 0) != 0:
            raise RuntimeError(f"{group}/{split} extraction failures: {report.get('failed', [])[:5]}")
        if report.get("total_requested") != expected_rows:
            raise RuntimeError(f"{group}/{split} requested {report.get('total_requested')} rows; expected {expected_rows}.")
        if report.get("written", 0) + report.get("skipped_cached", 0) != expected_rows:
            raise RuntimeError(f"{group}/{split} cache completion count mismatch.")

        run_cli(
            "build-group-matrix", "--config", CONFIG_ARG, "--splits", split,
            "--feature-group", group, "--cache-root", SMOKE_CACHE,
            "--output-dir", SMOKE_GROUP_DIR, "--stats", *AGGREGATION_STATS, "--overwrite",
        )
        group_dir = SMOKE_GROUP_DIR / group
        X_path, y_path = group_dir / f"X_{split}.npy", group_dir / f"y_{split}.npy"
        X = np.load(X_path, mmap_mode="r")
        y = np.load(y_path, mmap_mode="r")
        expected_shape = (expected_rows, EXPECTED_GROUP_WIDTHS[group])
        if X.shape != expected_shape or len(y) != expected_rows:
            raise ValueError(f"{group}/{split}: got X={X.shape}, y={len(y)}; expected X={expected_shape}.")
        extraction_reports[f"{group}/{split}"] = {
            "shape": list(X.shape), "written": report.get("written"),
            "seconds_including_aggregation": round(time.time() - start, 2),
        }
        del X, y
        # The smoke workspace is disposable, so remove frame caches after the aggregate matrix passes checks.
        cache_split = SMOKE_CACHE / "features" / group / split
        if cache_split.exists():
            shutil.rmtree(cache_split)

print(json.dumps(extraction_reports, indent=2))
print("PASS: all real feature extractors and aggregation commands completed for the sampled utterances.")

## 7. Run the real fusion command and validate feature dimensions / row alignment

In [ ]:
run_cli(
    "combine-fusion", "--config", CONFIG_ARG,
    "--splits", "train", "dev", "eval",
    "--feature-groups", *FEATURE_GROUPS,
    "--group-feature-dir", SMOKE_GROUP_DIR,
    "--output-dir", SMOKE_FUSION_DIR,
    "--overwrite",
)

fusion_schema = json.loads((SMOKE_FUSION_DIR / "feature_schema.json").read_text(encoding="utf-8"))
if int(fusion_schema.get("n_features", -1)) != 1350:
    raise ValueError(f"Expected 1,350 fused features, got {fusion_schema.get('n_features')}.")
fusion_shapes = {}
for split in ("train", "dev", "eval"):
    X = np.load(SMOKE_FUSION_DIR / f"X_{split}.npy", mmap_mode="r")
    y = np.load(SMOKE_FUSION_DIR / f"y_{split}.npy", mmap_mode="r")
    metadata = pd.read_csv(SMOKE_FUSION_DIR / f"metadata_{split}.csv")
    if X.shape != (smoke_manifest_summary[split]["n"], 1350):
        raise ValueError(f"Unexpected fused {split} shape: {X.shape}")
    if len(y) != len(metadata) or len(metadata) != len(X):
        raise ValueError(f"Fused {split} rows/labels/metadata are misaligned.")
    fusion_shapes[split] = list(X.shape)
    del X, y
print("Fusion shapes:", fusion_shapes)
print("PASS: actual fusion code produced aligned 1,350-feature matrices.")

## 8. Train all E1–E7 ablation models through the existing CLI

This uses the same feature-group combinations and the same `train-xgb` implementation as the full notebook. The estimator count and early-stopping patience are intentionally reduced for a quick integration test.

In [ ]:
ABLATIONS = {
    "E1_CQCC": ["cqcc"],
    "E2_TKEO": ["tkeo"],
    "E3_ACOUSTIC": ["acoustic"],
    "E4_CQCC_TKEO": ["cqcc", "tkeo"],
    "E5_CQCC_ACOUSTIC": ["cqcc", "acoustic"],
    "E6_TKEO_ACOUSTIC": ["tkeo", "acoustic"],
    "E7_ALL": ["cqcc", "tkeo", "acoustic"],
}

ablation_smoke_results = {}
for experiment, groups in ABLATIONS.items():
    model_dir = SMOKE_ABLATION_DIR / experiment
    run_cli(
        "train-xgb", "--config", CONFIG_ARG,
        "--feature-groups", *groups,
        "--feature-dir", SMOKE_FUSION_DIR,
        "--model-dir", model_dir,
        "--n-estimators", SMOKE_N_ESTIMATORS,
        "--early-stopping-rounds", SMOKE_EARLY_STOPPING,
        "--max-depth", 5, "--learning-rate", 0.03,
        "--n-jobs", 1, "--device", XGB_DEVICE,
    )
    required = [model_dir / "xgb_fusion.json", model_dir / "metrics.json",
                model_dir / "dev_predictions.csv", model_dir / "eval_predictions.csv"]
    missing = [str(p) for p in required if not p.is_file()]
    if missing:
        raise FileNotFoundError(f"{experiment} did not produce expected artifacts:\n" + "\n".join(missing))
    metrics = json.loads((model_dir / "metrics.json").read_text(encoding="utf-8"))
    ablation_smoke_results[experiment] = {
        "n_features": metrics.get("n_features"),
        "device": metrics.get("effective_device"),
        "best_iteration": metrics.get("best_iteration"),
        "eval_eer_percent_smoke_only": metrics.get("eval", {}).get("eer_percent"),
    }

print(json.dumps(ablation_smoke_results, indent=2))
print("PASS: all seven real E1–E7 train-xgb invocations completed.")

## 9. Exercise the official PA min-tDCF and integration scripts

The scripts are run against the smoke predictions and the official ASV score files. The values are **not benchmark-valid** because the CM side contains only a subset of utterances; this stage checks code integration and output schemas.

In [ ]:
SMOKE_TDCF_DIR.mkdir(parents=True, exist_ok=True)
run_tool(
    "tools/compute_pa_tdcf.py",
    "--project-root", SMOKE_ROOT,
    "--asv-dir", ASV_SCORE_DIR,
    "--output-dir", SMOKE_TDCF_DIR,
)
run_tool(
    "tools/integrate_tdcf_results.py",
    "--project-root", SMOKE_ROOT,
    "--tdcf-csv", SMOKE_TDCF_DIR / "pa_min_tdcf_summary.csv",
    "--no-backup",
)
comparison_path = SMOKE_TDCF_DIR / "ablation_eer_tdcf_comparison.csv"
comparison = pd.read_csv(comparison_path)
if set(comparison["experiment"].unique()) != set(ABLATIONS):
    raise ValueError("tDCF integration did not produce all seven ablation experiments.")
print(comparison[["experiment", "split", "eer_percent", "min_tdcf"]].to_string(index=False))
print("PASS: official tDCF and ablation-integration scripts executed on smoke outputs.")

## 10. Exercise E7_FINAL tuning with one trial and reduced tree count

This invokes the same `tools/tune_e7.py` script as the full notebook. Only the search budget is reduced: one trial and 80 maximum estimators instead of 80 trials and 10,000 maximum estimators. It still trains from the smoke train matrix, uses smoke dev for selection, and does not use eval for tuning.

In [ ]:
run_tool(
    "tools/tune_e7.py",
    "--feature-dir", SMOKE_FUSION_DIR,
    "--asv-dir", ASV_SCORE_DIR,
    "--output-dir", SMOKE_TUNING_DIR,
    "--trials", SMOKE_TUNING_TRIALS,
    "--seed", RANDOM_SEED,
    "--n-estimators", SMOKE_TUNING_N_ESTIMATORS,
    "--early-stopping-rounds", SMOKE_TUNING_EARLY_STOPPING,
    "--n-jobs", 1,
    "--device", XGB_DEVICE,
    "--verbose", 0,
)

TUNING_ARTIFACTS = [
    SMOKE_TUNING_DIR / "search_manifest.json",
    SMOKE_TUNING_DIR / "trials.csv",
    SMOKE_MODEL_DIR / "xgb_fusion.json",
    SMOKE_MODEL_DIR / "metrics.json",
    SMOKE_MODEL_DIR / "feature_names.json",
]
missing = [str(p) for p in TUNING_ARTIFACTS if not p.is_file()]
if missing:
    raise FileNotFoundError("E7_FINAL smoke tuning outputs missing:\n" + "\n".join(missing))
print("PASS: E7_FINAL tuning script completed and saved a model.")

## 11. Run the actual final evaluation script on the smoke eval matrix

In [ ]:
SMOKE_FINAL_EVAL_DIR = SMOKE_MODEL_DIR / "final_eval"
run_tool(
    "tools/evaluate_tuned_e7.py",
    "--project-root", SMOKE_ROOT,
    "--feature-dir", SMOKE_FUSION_DIR,
    "--model-dir", SMOKE_MODEL_DIR,
    "--asv-dir", ASV_SCORE_DIR,
    "--output-dir", SMOKE_FINAL_EVAL_DIR,
)

SMOKE_EVAL_METRICS = SMOKE_FINAL_EVAL_DIR / "eval_metrics.json"
SMOKE_EVAL_PREDICTIONS = SMOKE_FINAL_EVAL_DIR / "eval_predictions.csv"
if not SMOKE_EVAL_METRICS.is_file() or not SMOKE_EVAL_PREDICTIONS.is_file():
    raise FileNotFoundError("E7_FINAL smoke evaluation did not create both expected files.")
smoke_eval = json.loads(SMOKE_EVAL_METRICS.read_text(encoding="utf-8"))
print(json.dumps({
    "n_eval": smoke_eval.get("n_eval"),
    "n_features": smoke_eval.get("n_features"),
    "eer_percent_smoke_only": smoke_eval.get("threshold_independent_metrics", {}).get("eer_percent"),
    "min_tdcf_smoke_only": smoke_eval.get("official_min_tdcf", {}).get("value"),
    "fixed_threshold": smoke_eval.get("fixed_threshold", {}).get("value"),
}, indent=2))
print("PASS: final evaluation code completed on the smoke eval matrix.")

## 12. Run global and class-conditional SHAP on a small evaluation sample

In [ ]:
run_tool(
    "tools/shap_global_e7.py",
    "--project-root", SMOKE_ROOT,
    "--feature-dir", SMOKE_FUSION_DIR,
    "--model-dir", SMOKE_MODEL_DIR,
    "--output-dir", SMOKE_GLOBAL_SHAP_DIR,
    "--sample-size", 12,
    "--batch-size", 6,
    "--seed", RANDOM_SEED,
    "--top-k", 10,
)
run_tool(
    "tools/shap_class_conditional_e7.py",
    "--project-root", SMOKE_ROOT,
    "--feature-dir", SMOKE_FUSION_DIR,
    "--model-dir", SMOKE_MODEL_DIR,
    "--output-dir", SMOKE_CLASS_SHAP_DIR,
    "--samples-per-class", 6,
    "--batch-size", 6,
    "--seed", RANDOM_SEED,
    "--top-k", 10,
)

shap_expected = [
    SMOKE_GLOBAL_SHAP_DIR / "shap_global_group_importance.csv",
    SMOKE_CLASS_SHAP_DIR / "shap_class_conditional_group_importance.csv",
    SMOKE_CLASS_SHAP_DIR / "shap_bonafide_feature_importance.csv",
    SMOKE_CLASS_SHAP_DIR / "shap_spoof_feature_importance.csv",
]
missing = [str(p) for p in shap_expected if not p.is_file()]
if missing:
    raise FileNotFoundError("SHAP smoke outputs missing:\n" + "\n".join(missing))
print("PASS: global and class-conditional SHAP scripts executed.")

## 13. Run the individual explanation code on one sampled evaluation recording

In [ ]:
if RUN_INDIVIDUAL_SHAP:
    sample_eval_metadata = pd.read_csv(SMOKE_FUSION_DIR / "metadata_eval.csv", dtype={"utterance_id": str})
    test_utterance_id = str(sample_eval_metadata.iloc[0]["utterance_id"])
    run_tool(
        "tools/explain_one_trial.py",
        "--feature-dir", SMOKE_FUSION_DIR,
        "--model-dir", SMOKE_MODEL_DIR,
        "--prediction-file", SMOKE_EVAL_PREDICTIONS,
        "--output-dir", SMOKE_INDIVIDUAL_SHAP_DIR,
        "--utterance-id", test_utterance_id,
        "--top-k", 10,
    )
    individual_output = SMOKE_INDIVIDUAL_SHAP_DIR / f"{test_utterance_id}_group_contributions.csv"
    if not individual_output.is_file():
        raise FileNotFoundError(individual_output)
    print("PASS: individual SHAP explanation completed for", test_utterance_id)
else:
    print("Individual SHAP test skipped by RUN_INDIVIDUAL_SHAP=False.")

## 14. Run the evaluation-condition and paired-statistics scripts

These post-hoc scripts also run on the sampled evaluation subset. Their output is only a pipeline test.

In [ ]:
SMOKE_CONDITION_DIR = SMOKE_ROOT / "results" / "condition_analysis"
run_tool("tools/ASVspoof_PA_condition_analysis.py", "--root", SMOKE_ROOT)
joined_files = [SMOKE_CONDITION_DIR / f"{exp}_joined_eval_predictions.csv" for exp in ABLATIONS]
missing = [str(p) for p in joined_files if not p.is_file()]
if missing:
    raise FileNotFoundError("Condition analysis outputs missing:\n" + "\n".join(missing))

if RUN_PAIRED_CONDITION_STATS:
    run_tool(
        "tools/paired_pa_condition_stats.py",
        "--input-dir", SMOKE_CONDITION_DIR,
        "--output-dir", SMOKE_ROOT / "results" / "paired_condition_statistics",
        "--bootstrap", 200,
        "--seed", RANDOM_SEED,
    )
    paired_summary = SMOKE_ROOT / "results" / "paired_condition_statistics" / "paired_stats_overall.csv"
    if not paired_summary.is_file():
        raise FileNotFoundError(paired_summary)
print("PASS: condition analysis and paired statistics completed.")

## 15. Smoke-test report and interpretation

A successful final cell means the sampled utterances passed through the same existing feature-extraction, aggregation, fusion, ablation, tuning, evaluation and explainability code paths that the full notebook invokes.

It does **not** establish that the full corpus will run without resource problems, that the 80-trial search will complete, or that smoke metrics match the paper. The figure/table generator is intentionally not run here because it expects the four specific full-evaluation case IDs used in the manuscript. Run it in the complete reproducibility workflow after a full evaluation.

In [ ]:
summary = {
    "status": "SMOKE TEST COMPLETED",
    "smoke_root": str(SMOKE_ROOT),
    "samples_per_class_per_split": SAMPLES_PER_CLASS_PER_SPLIT,
    "manifests": smoke_manifest_summary,
    "feature_matrices": fusion_shapes,
    "ablation_experiments": list(ablation_smoke_results),
    "tuning_trials": SMOKE_TUNING_TRIALS,
    "xgb_device": XGB_DEVICE,
    "checked_stages": [
        "existing CLI manifest", "CQCC/TKEO/acoustic feature extraction",
        "frame-cache aggregation", "fusion", "E1-E7 XGBoost training",
        "official PA min-tDCF toolchain", "E7_FINAL tuning", "final evaluation",
        "global SHAP", "class-conditional SHAP", "individual SHAP",
        "condition analysis", "paired condition statistics",
    ],
    "warning": "Subset metrics are diagnostic only and must not be reported as benchmark results.",
}
report_path = SMOKE_ROOT / "smoke_test_report.json"
report_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
SMOKE_CONFIG_PATH.unlink(missing_ok=True)
print(json.dumps(summary, indent=2))
print("Smoke-test report:", report_path)
print("Temporary config removed:", not SMOKE_CONFIG_PATH.exists())

In [ ]:

for experiment in ABLATIONS:
    for split in ("dev", "eval"):
        path = (
            SMOKE_ABLATION_DIR
            / experiment
            / f"{split}_predictions.csv"
        )

        df = pd.read_csv(path)
        scores = pd.to_numeric(
            df["score_bonafide"], errors="coerce"
        )

        print(
            f"{experiment:20s} {split:5s} | "
            f"unique scores={scores.nunique()} | "
            f"NaN={scores.isna().sum()} | "
            f"min={scores.min():.6f} | "
            f"max={scores.max():.6f}"
        )
